# 04 · Build the Excel workbook

The assignment asks for every cash-flow step to be visible. This notebook writes
`excel/FN_FS9548_MBS_Pricing_Model.xlsx`, a corrected version of the original Excel model with the same
sheet layout:

* **Path 1 is rebuilt entirely with Excel formulas** (rate path → PSA → cash flows → discount factors → PV →
  expected cash flows). It uses the same random shocks and calibrated drift as Python, so the Excel path-1 price
  must equal the Python path-1 price. A check cell confirms it.
* The 10,000-path Monte Carlo results and the sensitivity tables are written as values from notebooks 02–03
  (10,000 paths × 52 months of formulas would make the workbook unusably slow).
* There is no `RAND()`, so the workbook no longer changes every time it recalculates.

Run notebooks 01–03 first.

In [1]:
import sys
import json
import shutil
import subprocess
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

import mbs_model as m

case = m.build_base_case()
inputs, paths = case["inputs"], case["paths"]
T, B0 = inputs["wam_months"], inputs["current_balance"]
results = json.loads((m.OUTPUT_DIR / "results.json").read_text())
p1, p2 = results["part1"], results["part2"]
agency_prices = np.loadtxt(m.OUTPUT_DIR / "agency_path_prices_full.csv", skiprows=1)
private_prices = np.loadtxt(m.OUTPUT_DIR / "private_label_path_prices_full.csv", skiprows=1)
_, py_prices, _ = m.value_agency(case)
_, py_private, _, _ = m.value_private_label(case)

OUT = m.PROJECT_ROOT / "excel" / "FN_FS9548_MBS_Pricing_Model.xlsx"

BOLD = Font(bold=True)
HEAD = PatternFill("solid", fgColor="DCE6F1")
INPUT = PatternFill("solid", fgColor="FFF2CC")


def header(ws, row, labels, col=1):
    for j, lab in enumerate(labels):
        c = ws.cell(row, col + j, lab)
        c.font = BOLD
        c.fill = HEAD
        c.alignment = Alignment(wrap_text=True, vertical="top")


def widths(ws, w=16, first=34):
    ws.column_dimensions["A"].width = first
    for j in range(2, ws.max_column + 2):
        ws.column_dimensions[get_column_letter(j)].width = w


wb = Workbook()

## Inputs

In [2]:
ws = wb.active
ws.title = "Inputs"
header(ws, 1, ["Input", "Value", "Source / note"])
rows = [
    ("Security", inputs["security"], "Bloomberg DES"),
    ("CUSIP", inputs["cusip"], "Bloomberg DES"),
    ("Pool type", inputs["pool_type"], "Bloomberg DES"),
    ("Pass-through coupon", inputs["coupon"], "Bloomberg DES"),
    ("WAC", inputs["wac"], "Bloomberg DES"),
    ("WAM / WARM (months)", inputs["wam_months"], "Bloomberg DES"),
    ("WALA (months)", inputs["wala_months"], "Bloomberg DES"),
    ("Original amount", inputs["orig_amount"], "Bloomberg DES"),
    ("Current balance B0", inputs["current_balance"], "Bloomberg DES (04/2026 factor)"),
    ("Pool factor", "=B10/B9", "Current / original; Bloomberg shows 0.57013048"),
    ("Base PSA", inputs["bloomberg_prepay_psa"], "Bloomberg BAM projection (BVAL/ALLQ screen)"),
    ("Refinancing sensitivity beta", m.BASE["beta"], "PSA multiplier = 1 - beta x (rate change / 1%)"),
    ("Multiplier floor", 0.75, "Lock-in floor"),
    ("Multiplier cap", 1.75, "Refinancing cap"),
    ("BVAL bid (32nds)", inputs["bval_bid_32nds"], "97-23 5/8 bid, 30 Apr 2026, NY 4pm"),
    ("BVAL bid (decimal, clean)", "=97+B16/32", ""),
    ("Accrued days (30/360)", inputs["accrued_days_30_360"], "1 Apr to 30 Apr settlement"),
    ("Accrued interest per 100", "=B5/12*B18/30*100", ""),
    ("BVAL bid full price", "=B17+B19", "Clean + accrued"),
    ("Payment lag (months after month start)", inputs["payment_lag_months_after_month_start"],
     "Paid on the 25th of the following month (54-day delay)"),
    ("Bloomberg BAM TOAS (bp)", inputs["bloomberg_bam_toas_bp"], "Bloomberg ALLQ / DES header"),
    ("Monte Carlo paths", m.BASE["n_paths"], "5,000 antithetic pairs, seed 2026 (Python)"),
]
for i, (k, v, note) in enumerate(rows, start=2):
    ws.cell(i, 1, k)
    c = ws.cell(i, 2, v)
    if not (isinstance(v, str) and v.startswith("=")):
        c.fill = INPUT
    ws.cell(i, 3, note)
widths(ws, 22)
ws.column_dimensions["C"].width = 55
NAMES = {"coupon": "Inputs!$B$5", "wac": "Inputs!$B$6", "wam": "Inputs!$B$7", "wala": "Inputs!$B$8",
         "B0": "Inputs!$B$10", "psa": "Inputs!$B$12", "beta": "Inputs!$B$13", "lo": "Inputs!$B$14",
         "hi": "Inputs!$B$15", "bid": "Inputs!$B$17", "ai": "Inputs!$B$19", "lag": "Inputs!$B$21"}

## Rate volatility and yield curve

In [3]:
ws = wb.create_sheet("Rate Volatility")
gt10 = case["gt10"]
gt10 = gt10[gt10.index <= pd.Timestamp(inputs["valuation_date"])]
header(ws, 1, ["Month-end", "GT10 yield (%)", "Monthly change (%)", "", "Statistic", "Value"])
for i, (d, y) in enumerate(gt10.items(), start=2):
    ws.cell(i, 1, d.to_pydatetime()).number_format = "yyyy-mm-dd"
    ws.cell(i, 2, float(y))
    if i > 2:
        ws.cell(i, 3, f"=B{i}-B{i-1}")
last = 1 + len(gt10)
ws["E2"], ws["F2"] = "Monthly volatility (% points)", f"=_xlfn.STDEV.S(C3:C{last})"
ws["E3"], ws["F3"] = "Monthly volatility (decimal) sigma", "=F2/100"
ws["E4"], ws["F4"] = "Observations (changes)", f"=COUNT(C3:C{last})"
ws["E5"] = "Excludes the 5 May 2026 print in the raw export (after the valuation date)."
widths(ws, 16, 14)
ws.column_dimensions["E"].width = 34
SIGMA = "'Rate Volatility'!$F$3"

ws = wb.create_sheet("Yield Curve")
zero = case["zero_curve"]
header(ws, 1, ["Maturity (yrs)", "Par yield (BEY)", "Discount factor", "Zero rate (cont.)", "Zero rate (BEY)"])
for i, r in enumerate(zero.itertuples(), start=2):
    for j, v in enumerate([r.years, r.par_yield, r.discount_factor, r.zero_cc, r.zero_bey], start=1):
        ws.cell(i, j, float(v))
ws["G1"] = ("Bills (<=1y) treated as zero yields; 1.5y-5y bootstrapped from linearly interpolated par yields: "
            "1 = c*sum(P) + (1+c)*P(t_n). Monthly P(0,t) by linear interpolation of continuous zero rates.")
header(ws, 15, ["Month t", "P(0,t) Treasury", "Mean path DF (Python, 10,000 paths)", "Calibrated drift theta_t"])
D = m.path_discount_factors(paths.rates)
for t in range(T + 1):
    ws.cell(16 + t, 1, t)
    ws.cell(16 + t, 2, float(paths.market_df[t]))
    ws.cell(16 + t, 3, 1.0 if t == 0 else float(D[:, t - 1].mean()))
    if t < T:
        ws.cell(16 + t, 4, float(paths.theta[t]))
widths(ws, 18, 14)

## PSA model

In [4]:
ws = wb.create_sheet("PSA Model")
header(ws, 1, ["Projection month", "Loan age", "100 PSA CPR", "Base PSA CPR", "Base SMM"])
for t in range(1, T + 1):
    r = t + 1
    ws.cell(r, 1, t)
    ws.cell(r, 2, f"={NAMES['wala']}+A{r}")
    ws.cell(r, 3, f"=0.06*MIN(B{r},30)/30")
    ws.cell(r, 4, f"=C{r}*{NAMES['psa']}")
    ws.cell(r, 5, f"=1-(1-D{r})^(1/12)")
widths(ws, 16, 18)

## Rate path 1 (formulas)

In [5]:
ws = wb.create_sheet("Rate Paths")
header(ws, 1, ["Month k", "Shock eps_k (fixed, from Python seed 2026)", "W_k = sum eps",
               "theta_k (calibrated drift)", "Rate shock sigma*W_k", "Short rate r_k = MAX(0, theta+sigma*W)"])
eps_k = np.concatenate([[0.0], paths.eps[0, : T - 1]])
for k in range(T):
    r = k + 2
    ws.cell(r, 1, k)
    ws.cell(r, 2, float(eps_k[k])).fill = INPUT
    ws.cell(r, 3, "=0" if k == 0 else f"=C{r-1}+B{r}")
    ws.cell(r, 4, float(paths.theta[k])).fill = INPUT
    ws.cell(r, 5, f"={SIGMA}*C{r}")
    ws.cell(r, 6, f"=MAX(0,D{r}+E{r})")
ws["H1"] = ("r_k applies from month k to k+1. theta_k is solved in Python so that the average discount factor "
            "over 10,000 paths equals the Treasury P(0,k+1) (see Yield Curve sheet).")
widths(ws, 20, 10)

## Agency cash flows on path 1 (formulas)

In [6]:
ws = wb.create_sheet("Agency CF")
cols = ["Month", "Beginning balance", "Remaining term", "Scheduled payment (WAC)", "Gross interest (WAC)",
        "Investor interest (PTR)", "Scheduled principal", "Rate shock at month start", "PSA multiplier",
        "CPR", "SMM", "Prepayment", "Total principal", "Ending balance", "Total cash flow",
        "Short rate r_(t-1)", "DF to end of month t-1", "DF on payment date", "Present value"]
header(ws, 1, cols)
for t in range(1, T + 1):
    r = t + 1
    rp = t + 1  # Rate Paths row for k = t-1
    ws.cell(r, 1, t)
    ws.cell(r, 2, f"={NAMES['B0']}" if t == 1 else f"=N{r-1}")
    ws.cell(r, 3, f"={NAMES['wam']}-A{r}+1")
    ws.cell(r, 4, f"=B{r}*({NAMES['wac']}/12)/(1-(1+{NAMES['wac']}/12)^(-C{r}))")
    ws.cell(r, 5, f"=B{r}*{NAMES['wac']}/12")
    ws.cell(r, 6, f"=B{r}*{NAMES['coupon']}/12")
    ws.cell(r, 7, f"=MIN(D{r}-E{r},B{r})")
    ws.cell(r, 8, f"='Rate Paths'!E{rp}")
    ws.cell(r, 9, f"=MIN({NAMES['hi']},MAX({NAMES['lo']},1-{NAMES['beta']}*H{r}/0.01))")
    ws.cell(r, 10, f"=MIN('PSA Model'!D{r}*I{r},0.999)")
    ws.cell(r, 11, f"=1-(1-J{r})^(1/12)")
    ws.cell(r, 12, f"=K{r}*(B{r}-G{r})")
    ws.cell(r, 13, f"=G{r}+L{r}")
    ws.cell(r, 14, f"=MAX(B{r}-M{r},0)")
    ws.cell(r, 15, f"=F{r}+M{r}")
    ws.cell(r, 16, f"='Rate Paths'!F{rp}")
    ws.cell(r, 17, "=1" if t == 1 else f"=Q{r-1}/(1+P{r-1}/12)")
    ws.cell(r, 18, f"=Q{r}*(1+P{r}/12)^(-{NAMES['lag']})")
    ws.cell(r, 19, f"=O{r}*R{r}")
end = T + 1
s = end + 2
ws.cell(s, 17, "Total PV ($)").font = BOLD
ws.cell(s, 19, f"=SUM(S2:S{end})")
ws.cell(s + 1, 17, "Path 1 full price per 100").font = BOLD
ws.cell(s + 1, 19, f"=S{s}/{NAMES['B0']}*100")
ws.cell(s + 2, 17, "Path 1 clean price per 100").font = BOLD
ws.cell(s + 2, 19, f"=S{s+1}-{NAMES['ai']}")
ws.cell(s + 3, 17, "Python path 1 full price").font = BOLD
ws.cell(s + 3, 19, float(py_prices[0]))
ws.cell(s + 4, 17, "Check: Excel − Python (should be ~0)").font = BOLD
ws.cell(s + 4, 19, f"=S{s+1}-S{s+3}")
ws.cell(s + 6, 1, ("One path only. The model value is the average over 10,000 paths: see Agency MC Output. "
                   "DF on payment date = DF(end of month t-1) x (1+r_(t-1)/12)^(-25/30)."))
widths(ws, 15, 8)
AGENCY_CHECK_ROW = s + 4

## Credit risk and private-label cash flows on path 1 (formulas)

In [7]:
ws = wb.create_sheet("Credit Risk")
header(ws, 1, ["Input", "Value", "Note"])
cr = [("Guarantor rating", "A (S&P)", "Hypothetical private guarantor"),
      ("Annual probability of default", p2["pd_annual"], "S&P long-run 'A' default experience, ~0.1%/yr over 4-5 yrs"),
      ("Recovery rate", p2["recovery"], "Senior unsecured assumption"),
      ("Loss given default", "=1-B4", ""),
      ("Monthly PD", "=1-(1-B3)^(1/12)", "")]
for i, (k, v, n) in enumerate(cr, start=2):
    ws.cell(i, 1, k)
    c = ws.cell(i, 2, v)
    if i in (3, 4):
        c.fill = INPUT
    ws.cell(i, 3, n)
header(ws, 1, ["Month t", "Survival S_t", "Marginal default S_(t-1) - S_t"], col=5)
ws.cell(2, 5, 0)
ws.cell(2, 6, 1)
for t in range(1, T + 1):
    r = t + 2
    ws.cell(r, 5, t)
    ws.cell(r, 6, f"=(1-$B$3)^(E{r}/12)")
    ws.cell(r, 7, f"=F{r-1}-F{r}")
widths(ws, 16, 30)
ws.column_dimensions["C"].width = 50

ws = wb.create_sheet("Private CF")
header(ws, 1, ["Month", "Promised cash flow (Agency CF)", "Beginning balance B_(t-1)", "Survival S_t",
               "Marginal default", "Expected cash flow", "DF on payment date", "Credit-adjusted PV"])
for t in range(1, T + 1):
    r = t + 1
    ws.cell(r, 1, t)
    ws.cell(r, 2, f"='Agency CF'!O{r}")
    ws.cell(r, 3, f"='Agency CF'!B{r}")
    ws.cell(r, 4, f"='Credit Risk'!F{r+1}")
    ws.cell(r, 5, f"='Credit Risk'!G{r+1}")
    ws.cell(r, 6, f"=D{r}*B{r}+E{r}*'Credit Risk'!$B$4*C{r}")
    ws.cell(r, 7, f"='Agency CF'!R{r}")
    ws.cell(r, 8, f"=F{r}*G{r}")
ws.cell(s, 7, "Total PV ($)").font = BOLD
ws.cell(s, 8, f"=SUM(H2:H{end})")
ws.cell(s + 1, 7, "Path 1 full price per 100").font = BOLD
ws.cell(s + 1, 8, f"=H{s}/{NAMES['B0']}*100")
ws.cell(s + 2, 7, "Path 1 clean price per 100").font = BOLD
ws.cell(s + 2, 8, f"=H{s+1}-{NAMES['ai']}")
ws.cell(s + 3, 7, "Python path 1 full price").font = BOLD
ws.cell(s + 3, 8, float(py_private[0]))
ws.cell(s + 4, 7, "Check: Excel − Python (should be ~0)").font = BOLD
ws.cell(s + 4, 8, f"=H{s+1}-H{s+3}")
ws.cell(s + 6, 1, "E[CF_t] = S_t x CF_t + (S_(t-1) - S_t) x R x B_(t-1): recovery is on the outstanding principal claim.")
widths(ws, 18, 8)

## Monte Carlo output (values from Python)

In [8]:
for name, prices in [("Agency MC Output", agency_prices), ("Private MC Output", private_prices)]:
    ws = wb.create_sheet(name)
    header(ws, 1, ["Path", "Full price per 100", "", "Summary", "Value"])
    n = len(prices)
    h = n // 2
    ws.cell(1, 3, "Antithetic pair average (path i and i+5,000)").font = BOLD
    for i, v in enumerate(prices, start=2):
        ws.cell(i, 1, i - 1)
        ws.cell(i, 2, float(v))
        if i - 1 <= h:
            ws.cell(i, 3, f"=(B{i}+B{i+h})/2")
    summ = [("Average full price", f"=AVERAGE(B2:B{n+1})"),
            ("Accrued interest", f"={NAMES['ai']}"),
            ("Average clean price", "=E2-E3"),
            ("Std error (antithetic pairs)", f"=_xlfn.STDEV.S(C2:C{h+1})/SQRT({h})"),
            ("BVAL bid (clean)", f"={NAMES['bid']}"),
            ("Model − market", "=E4-E6"),
            ("Conclusion", '=IF(E7>0,"Model above market: cheap vs Treasuries at zero OAS",'
                           '"Model below market: rich")')]
    for j, (k, f) in enumerate(summ, start=2):
        ws.cell(j, 4, k)
        ws.cell(j, 5, f)
    ws["D10"] = "Paths 1-5,000 and 5,001-10,000 are antithetic pairs (shocks eps and -eps)."
    widths(ws, 18, 8)
    ws.column_dimensions["D"].width = 34

## Sensitivity and summary

In [9]:
ws = wb.create_sheet("Sensitivity")
r = 1
for title, table in [("PSA speed (same 10,000 paths)", p1["psa_sensitivity"]),
                     ("Refinancing sensitivity beta", p1["beta_sensitivity"]),
                     ("Rate volatility", p1["vol_sensitivity"])]:
    ws.cell(r, 1, title).font = BOLD
    cols = list(table[0].keys())
    header(ws, r + 1, cols)
    for i, rec in enumerate(table, start=r + 2):
        for j, c in enumerate(cols, start=1):
            ws.cell(i, j, rec[c])
    r += len(table) + 4
ws.cell(r, 1, "Private-label clean value: annual PD (rows) x recovery (columns)").font = BOLD
grid = pd.DataFrame(p2["credit_grid"])
header(ws, r + 1, ["Annual PD"] + list(grid.columns))
for i, (idx, rowv) in enumerate(grid.iterrows(), start=r + 2):
    ws.cell(i, 1, idx)
    for j, v in enumerate(rowv, start=2):
        ws.cell(i, j, float(v))
r += len(grid) + 4
ws.cell(r, 1, "Bridge from the original Excel method to the corrected value").font = BOLD
header(ws, r + 1, ["Step", "Value per 100"])
for i, rec in enumerate(p1["bridge"], start=r + 2):
    ws.cell(i, 1, rec["step"])
    ws.cell(i, 2, rec["value per 100"])
widths(ws, 18, 70)

ws = wb.create_sheet("Summary", 0)
header(ws, 1, ["Metric", "Value", "Interpretation"])
summary_rows = [
    ("Security", "=Inputs!B2", "Fannie Mae UMBS 15-year pool, 3.0% pass-through"),
    ("BVAL bid (clean)", f"={NAMES['bid']}", "30 Apr 2026"),
    ("PART I: AGENCY", None, None),
    ("Model value, clean (10,000 paths)", "='Agency MC Output'!E4", "178 PSA with refinancing overlay; Treasury-calibrated paths"),
    ("Monte Carlo standard error", "='Agency MC Output'!E5", "Simulation noise is negligible"),
    ("Model − market", "='Agency MC Output'!E7", "Positive: above market at zero OAS"),
    ("OAS at BVAL bid (bp)", p1["oas_bp_at_bid"], "Spread over simulated Treasury short rates that reprices the bid"),
    ("Bloomberg BAM TOAS (bp)", inputs["bloomberg_bam_toas_bp"], "Bloomberg's own model, for comparison"),
    ("Effective duration (yrs)", p1["effective_duration"], "±25 bp parallel curve shift, OAS held constant"),
    ("Conclusion", "Fair to slightly cheap",
     "OAS about 8 bp wider than Bloomberg's TOAS; varies ±10 bp across 100-300 PSA, so not certain"),
    ("PART II: PRIVATE LABEL (A-rated guarantor)", None, None),
    ("Annual PD / recovery", f"{p2['pd_annual']:.2%} / {p2['recovery']:.0%}", "See Credit Risk"),
    ("Model value, clean", "='Private MC Output'!E4", "Expected cash-flow approach"),
    ("Credit discount vs agency (points)", "=B5-B14", "Cost of replacing the government guarantee"),
    ("Model − market", "='Private MC Output'!E7", "Still above the agency BVAL bid under historical PD"),
    ("Break-even annual PD", p2["breakeven_pd_annual"], "PD at which private-label value = BVAL bid"),
    ("CHECKS", None, None),
    ("Excel vs Python path 1 (agency)", f"='Agency CF'!S{AGENCY_CHECK_ROW}", "Should be ~0"),
    ("Excel vs Python path 1 (private)", f"='Private CF'!H{AGENCY_CHECK_ROW}", "Should be ~0"),
]
for i, (k, v, note) in enumerate(summary_rows, start=2):
    ws.cell(i, 1, k)
    if v is None:
        ws.cell(i, 1).font = BOLD
        continue
    ws.cell(i, 2, v)
    ws.cell(i, 3, note)
widths(ws, 22, 42)
ws.column_dimensions["C"].width = 80

ws = wb.create_sheet("Corrections Log")
header(ws, 1, ["#", "Original model", "Correction"])
log = [
    ("Discount factor DF_t = 1/(1+r_t/12)^t used only the month-t rate as if it were a spot rate.",
     "Path-wise cumulative discount factor: product of 1/(1+r_k/12) over k = 0..t-1."),
    ("Rate paths were a driftless random walk started at the 10-year yield (4.387%), unrelated to the curve.",
     "One-month short-rate paths with drift calibrated so that average discount factors reprice the Treasury zero curve."),
    ("Cash flows were identical on every path (PSA only), so the simulation could not value the prepayment option.",
     "PSA speed responds to the simulated rate change (beta = 0.20, bounded 0.75x-1.75x)."),
    ("RAND() with 100 paths: results changed on every recalculation; s.e. about 0.13, so cheap/rich flipped across PSA cases.",
     "10,000 antithetic paths with a fixed seed; the same paths for every scenario."),
    ("Report quoted three different agency prices (97.5371, 97.4949) and market prices 97.7383 / 97.7393.",
     "Single set of reproducible numbers generated by the notebooks."),
    ("Clean BVAL price compared with the value of cash flows assumed paid at month-end.",
     "Cash flows discounted to the actual 25th-of-month payment dates; accrued interest subtracted before comparing."),
    ("Private-label expected CF = CF_t x (S_t + R x q_t): recovery only on one month's cash flow.",
     "E[CF_t] = S_t x CF_t + q_t x R x B_(t-1): recovery on the outstanding principal claim."),
    ("Private-label price (97.5371) came from a different random draw and was higher than the agency price.",
     "Both parts valued on identical paths; the private-label value is always below the agency value."),
    ("Volatility series included a 5-day change (30 Apr to 5 May 2026).",
     "Month-end changes only, through the 30 Apr 2026 valuation date."),
]
for i, (a, b) in enumerate(log, start=2):
    ws.cell(i, 1, i - 1)
    ws.cell(i, 2, a).alignment = Alignment(wrap_text=True, vertical="top")
    ws.cell(i, 3, b).alignment = Alignment(wrap_text=True, vertical="top")
ws.column_dimensions["A"].width = 4
ws.column_dimensions["B"].width = 70
ws.column_dimensions["C"].width = 70

OUT.parent.mkdir(exist_ok=True)
wb.save(OUT)
print("wrote", OUT.relative_to(m.PROJECT_ROOT))

wrote excel/FN_FS9548_MBS_Pricing_Model.xlsx


### Recalculate and verify
openpyxl writes formulas without cached values. If LibreOffice is installed the workbook is recalculated
headlessly so the values show in any viewer, and the path-1 checks are read back. Otherwise Excel recalculates
when the file is opened.

In [10]:
soffice = shutil.which("soffice") or shutil.which("libreoffice")
if soffice:
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run([soffice, "--headless", "--calc", "--convert-to", "xlsx", "--outdir", tmp, str(OUT)],
                       check=True, capture_output=True)
        shutil.copy(Path(tmp) / OUT.name, OUT)
    from openpyxl import load_workbook
    v = load_workbook(OUT, data_only=True)["Summary"]
    for row in v.iter_rows(min_row=2, values_only=True):
        if row[1] is not None:
            print(f"{row[0]:<40} {row[1]}")
else:
    print("LibreOffice not found: open the workbook in Excel to calculate it.")

Security                                 FN FS9548 Mtge
BVAL bid (clean)                         97.73828125
Model value, clean (10,000 paths)        98.1979884031903
Monte Carlo standard error               0.000269813058658177
Model − market                           0.459707153190337
OAS at BVAL bid (bp)                     25.669507147475
Bloomberg BAM TOAS (bp)                  17.9
Effective duration (yrs)                 1.79451532477478
Conclusion                               Fair to slightly cheap
Annual PD / recovery                     0.10% / 40%
Model value, clean                       98.0898538188764
Credit discount vs agency (points)       0.108134584313987
Model − market                           0.35157256887635
Break-even annual PD                     0.00426268926864252
Excel vs Python path 1 (agency)          0
Excel vs Python path 1 (private)         0
